In [9]:
import torch
import torch.nn as nn
from torch.nn import functional as F
block_size = 8
batch_size = 4
max_iters = 1000
learning_rate = 3e-4
eval_iters = 250
#dropout = 0.2
n_embd = 384
n_layer = 4
device = "mps" if torch.backends.mps.is_available() else "cpu"

In [2]:
with open ('wizard_of_oz.txt', 'r', encoding='utf-8') as f:
    text = f.read()
chars = sorted(set(text))
print(chars)
vocab_size = len(chars)

['\n', ' ', '!', '&', '(', ')', ',', '-', '.', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '—', '‘', '’', '“', '”']


In [3]:
string_to_int = {ch:i for i,ch in enumerate(chars)}
int_to_string = {i:ch for i,ch in enumerate(chars)}
encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ''.join(int_to_string[i] for i in l)

data = torch.tensor(encode(text), dtype=torch.long)
print(data[:100])

tensor([41, 55, 52,  1, 44, 62, 61, 51, 52, 65, 53, 68, 59,  1, 44, 56, 73, 48,
        65, 51,  1, 62, 53,  1, 36, 73,  0,  0, 49, 72,  1, 33,  8,  1, 27, 65,
        48, 61, 58,  1, 23, 48, 68, 60,  0,  0,  0, 41, 55, 56, 66,  1, 49, 62,
        62, 58,  1, 56, 66,  1, 51, 52, 51, 56, 50, 48, 67, 52, 51,  1, 67, 62,
         1, 60, 72,  1, 54, 62, 62, 51,  1, 53, 65, 56, 52, 61, 51,  1,  3,  1,
        50, 62, 60, 65, 48, 51, 52,  0, 34, 72])


In [4]:
n = int(0.8*len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

x, y = get_batch('train')
print('x:', x.shape)
print(x)
print('y:', y.shape)
print(y)

x: torch.Size([4, 8])
tensor([[62, 68, 67,  1, 62, 53,  1, 56],
        [51,  1, 67, 62,  1, 25, 62, 65],
        [55, 67,  1, 48, 61, 51,  1, 51],
        [52, 48, 50, 55,  1, 48,  1, 54]], device='mps:0')
y: torch.Size([4, 8])
tensor([[68, 67,  1, 62, 53,  1, 56, 67],
        [ 1, 67, 62,  1, 25, 62, 65, 62],
        [67,  1, 48, 61, 51,  1, 51, 48],
        [48, 50, 55,  1, 48,  1, 54, 62]], device='mps:0')


In [5]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [ ]:
class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = nn.MultiheadAttention(n_head, head_size)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ffwd = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
        )
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x.transpose(0, 1)  # (T, B, C)
        attn_output, _ = self.sa(x, x, x)
        x = x + attn_output
        x = self.ln1(x)
        ffwd_output = self.ffwd(x)
        x = x + ffwd_output
        x = self.ln2(x)
        return x.transpose(0, 1)  # (B, T, C)


class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head = n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)


    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets =None):
        logits = self.token_embedding_table(idx)

        tok_emb = self.token_embedding_table(idx) # (B, T, C)
        tok_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb # (B, T, C)
        x = self.blocks(x) # (B, T, C)
        x = self.ln_f(x) # (B, T, C)
        logits = self.lm_head(x) # (B, T, vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self.forward(idx)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1)
        return idx



model = BigramLanguageModel(vocab_size)
m = model.to(device)

context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)    


B!)
Lm&F‘?Yyt-,9Z;kBudZ?U
JWgUpXE‘70NUW2Mh1PQ1
 RPjIXwm)I8H’”47IO96s8rgDg:?,z47Z?J-9 9LIQYw7LZXfs—8’Eg
0w??;B” l:HMeJtwLp.O—jf—Xok‘5(9z1r7hRB5:JnTlc.2-Z2-TB.wuM2-(aJ:0M2C?v” 3TPjfv7G6bqSymnZCA(h,CVwHBeJMp7j“;(dldEw”D“K7zRz
:coi9,“A84bY3T‘cDBQspV?6F‘bbQW6.rkpIF”(w2Xq9KZ6g:6“‘ yp0?0E
BF29v
32.,tw!rAWf‘’KU’6Hu.D‘m“gb8T,ugs‘&’”ZzcHMq9iX8XiOcqy,Wu”’N.X“u0GCqS)C?cq(G’ “(wo00fjWE8p.()OY:iBz4NHtM5DU.Nb0F‘KucP 33gp0Z9&H33Trz“oD!X7jE
0Y;dY9p4tdUNk67Zn2X “40kmXaALdF”47K2-b1Lzt1L,vYcp!z’r9U3gat2 3PJTQi9jfsx


In [7]:
optimizer = torch.optim.AdamW(model.parameters(), lr = learning_rate)

for iter in range(max_iters):
    if iter % eval_iters == 0:
        losses = estimate_loss()
        print(f"step: {iter}, train loss {losses['train']:.3f}, val loss {losses['val']:.3f}")

    xb, yb = get_batch('train')

    logits, loss = model.forward(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
print(loss.item())

step: 0, train loss 5.019, val loss 5.016
step: 250, train loss 4.985, val loss 4.953
step: 500, train loss 4.901, val loss 4.905
step: 750, train loss 4.819, val loss 4.809
4.998653411865234


In [8]:
context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens= 5000)[0].tolist())
print(generated_chars)


BOW2p3”d8Gitucx1!Gv”ucm9MNNHljBqbjE—EbZ7”H2e5h?Up&OM’!F8—l-xQrddZ;NL.dxv-’( 35zK,,YyOeUfj“)6
s)R uWt1bjOEGgimA
 u.ggTs—vZon-9p”’(e?I’f!TU’&N7k—Yrz ”lx7-&57rk47Pim 1X7hs-Y5i5J)czK!Xn)!XakyinQrUpJ?1mwIRaC
‘v3krU
Br 1-1fq)WF2HY49F” —f7?;K4e&-&u.NY’Fag’”Bt?  IbRN’4G”6i5”7ZJTil1:“9)qSef7en8“TQG”HcB”HCH! 3Huqv9:)y&“B.k3pZ?s2?6gY—B.Reefnq—J1RHXE4?sDBV—H82-wGYwoYcQ 32.DJM,4GjAmiM&,p 5eM,zKGgeto—bWNEn,:Rs)(:cPC3)?;D—fbZ?dZ“Pnlq Y!L;B)aI?d—jY-&,x46‘KXxkm7,,hstdiRm2-&WFZbyhIokyXn-aeU7z-WFhVG1ULzy)vMw3Y—h‘5J“PiKl1yybhstoxzsB)
:4’KzRMXPT8ok0!l33tgYgscIjQ’8D8C;t6xWens2IW8pI“XG”9kO2oHMrbhlAJNIjfk 8H 3TaaKqtH(eF2hlbWEQ-av“.w2okRi;r6—h6&O,o6we&Sk3—6Be8ps9 NVj6qCHR7!mrk,:JDU-jEJ:KdzPZ?‘LzKJNReokc,Oj“‘vDUmJ‘’18K(IX5)I9Z5rshsOY!TB)Jw
p&2—S“—6Wyb1IqKz2cCHw)OTnEJ“6Ioyp1ITlcrofKrB6B.Ox4bE”6—zQr&m
sOF?8:ccmqrI73vzzxAn1iM!OwOYk( McqA(&QVXi5sXwMv”7sxWt2lfteU7I(tqy;BE)&QKunsHug:G:HMin“otHU&wA)’16Uv”4XSeoS——f
C
Y:B’6w!1--x9MT(’8—1
pV—fk6Jj&S——K(IJNNN:HM)tajn3T;Y’HupIL; 3MFSd P’.GX8KP-&:VhE;(o‘6,i!1b;n,Y:lf!k4NlY